In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
# author Alex J. Yang
import numpy as np
import pandas as pd
import itertools


In [ ]:
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
df = load_table('papers')


In [ ]:
df


In [ ]:
# calculate CI using bootstrap


In [ ]:
def yearGroup(x):
    if x<1920:
        return '1900-1919'
    elif x<1940:
        return '1920-1939'
    elif x<1960:
        return '1940-1959'
    elif x<1980:
        return '1960-1979'
    else:
        return '1980-1999'
    
def delayGroup(x):
    if x<2:
        return x
    elif x<10:
        return '2-9'
    elif x<20:
        return '10-19'
    elif x<50:
        return '20-49'
    else:
        return '50+'
df['yearGroup'] = df['Year'].apply(yearGroup)
df['delayGroup'] = df['SB_T'].apply(delayGroup)


In [ ]:
df1= df[(df['type_Concave'] == 0)&(df['type_Immediacy'] == 0)].copy()
df2 = df[df['type_Concave'] == 1].copy()
df3 = df[df['type_Immediacy'] == 1].copy()


In [ ]:
convert_column_to_percentile = percentile_column
convert_column_to_percentile(df1, 'SB_B')
def BpGroup(x):
    if x<5:
        return 5
    elif x<50:
        return 50
    elif x<95:
        return 95
    else:
        return 100
df1['SB_B_p'] = df1['SB_B_percentile'].apply(BpGroup)


In [ ]:
df.head(3)


In [ ]:
df1.head(3)


In [ ]:
df2.head(3)


In [ ]:
df3.head(3)


In [ ]:
df.columns


# Distribution


In [ ]:
d = pd.DataFrame (df1[df1['SB_B_p']==100].groupby('Year').count()['SB_B'])
d1 = pd.DataFrame(df1[df1['SB_B_p']==95].groupby('Year').count()['SB_B'])
d2 = pd.DataFrame(df1[df1['SB_B_p']==50].groupby('Year').count()['SB_B'])
d3 = pd.DataFrame(df1[df1['SB_B_p']==5].groupby('Year').count()['SB_B'])

f, ax = plt.subplots(figsize=(7, 7))
plt.plot( d.index ,  d.values ,linewidth=3, c = '#154360' ,ls='-', label = 'B 95-100')
plt.plot(d1.index , d1.values ,linewidth=3, c = '#1F618D' ,ls=':', label = 'B 50-95')
plt.plot(d2.index , d2.values ,linewidth=3, c = '#2874A6' ,ls='--', label = 'B 5-50')
plt.plot(d3.index , d3.values ,linewidth=3, c = '#3498DB' ,ls='-.', label = 'B 0-5')
plt.ylim(1.1,1000000)
plt.xlim(1900,2000)
plt.yticks(fontsize=30,)
plt.xticks(fontsize=25,)
plt.xlabel('year',fontsize = 30)
plt.ylabel('#paper',fontsize = 30)
plt.legend(fontsize = 25)
ax.set_yscale("log")
plt.show()


In [ ]:
d = pd.DataFrame(df.groupby('Year').count()['SB_B'])
d1 = pd.DataFrame(df1.groupby('Year').count()['SB_B'])
d2 = pd.DataFrame(df2.groupby('Year').count()['SB_B'])
d3 = pd.DataFrame(df3.groupby('Year').count()['SB_B'])

f, ax = plt.subplots(figsize=(10, 7))
plt.plot( d.index ,  d.values , c = '#424949' , label = 'total')
plt.plot(d1.index , d1.values , c = '#2471A3' , label = 'concave')
plt.plot(d2.index , d2.values , c = '#117A65' , label = 'convex')
plt.plot(d3.index , d3.values , c = '#A93226' , label = 'immediacy')
plt.ylim(1.1,100000000)
plt.xlim(1900,2000)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('year',fontsize = 20)
plt.ylabel('#paper',fontsize = 20)
plt.legend(fontsize = 20,loc='lower right')
ax.set_yscale("log")
plt.show()


In [ ]:
d = pd.DataFrame(  df.groupby('yearGroup').count()['SB_B'])
d1 = pd.DataFrame(df1.groupby('yearGroup').count()['SB_B'])
d2 = pd.DataFrame(df2.groupby('yearGroup').count()['SB_B'])
d3 = pd.DataFrame(df3.groupby('yearGroup').count()['SB_B'])

dd = pd.concat([d,d1,d2,d3],axis = 1)
plt.subplots(1,5,figsize=(40,7))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.05,hspace=0)
for _ in range(5):
    ax = plt.subplot(1,5, _+1)
    x = [1,2,3,4]
    clst = ['#424949','#2471A3','#117A65','#A93226']*5
    y = dd.iloc[_,:].values/1000
    plt.bar(x,y,color= clst)
    plt.yticks(fontsize=23,)
    plt.xticks(x,['total','concave','convex','immediacy'],fontsize=22,)
    plt.xlabel(dd.index[_],fontsize = 35)
    if _==0:
        plt.ylabel('#paper (thousand)',fontsize = 35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


In [ ]:
d = pd.DataFrame (df1[df1['SB_B_p']==100].groupby('yearGroup').count()['SB_B'])
d1 = pd.DataFrame(df1[df1['SB_B_p']==95].groupby ('yearGroup').count()['SB_B'])
d2 = pd.DataFrame(df1[df1['SB_B_p']==50].groupby ('yearGroup').count()['SB_B'])
d3 = pd.DataFrame(df1[df1['SB_B_p']==5].groupby  ('yearGroup').count()['SB_B'])

dd = pd.concat([d,d1,d2,d3],axis = 1)
for _ in range(5):
    f, ax = plt.subplots(figsize=(5, 7))
    x = [1,2,3,4]
    clst =  ['#154360','#1F618D','#2874A6','#3498DB']*5
    y = dd.iloc[_,:].values/1000
    plt.bar(x,y,color= clst)
    plt.yticks(fontsize=23,)
    plt.xticks(x,['\n95-100','50-95','5-50','0-5'],fontsize=22,)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.show()


In [ ]:
d =  pd.DataFrame (df.groupby('Fieldname').count()['SB_B'])
d1 = pd.DataFrame(df1.groupby('Fieldname').count()['SB_B'])
d2 = pd.DataFrame(df2.groupby('Fieldname').count()['SB_B'])
d3 = pd.DataFrame(df3.groupby('Fieldname').count()['SB_B'])
dlst = [d,d1,d2,d3]
clst = ['#424949','#2471A3','#117A65','#A93226']
labellst = ['total','concave','convex','immediacy']
plt.subplots(1,4,figsize=(7,7))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.15,hspace=0)
for _ in range(4):
    ax = plt.subplot(1,4, _+1)
    plt.barh( dlst[_].index ,  width=np.concatenate(dlst[_].values) , color = clst[_])
    ax.set_xscale("log")
    plt.xlim(1,1000000)
    plt.ylim(-.7,18.7)
    if _>0:
        plt.yticks(fontsize=0,)
    else:
        plt.yticks(fontsize=15,)
    plt.xticks(fontsize=10,)
    plt.xlabel(labellst[_],fontsize = 19)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


In [ ]:
f, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(data=df, x="Year", y="SB_T",label=r'$𝑡_𝛼$')
sns.lineplot(data=df, x="Year", y="SB_B",label='B')
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('year',fontsize = 20)
plt.ylabel('mean value',fontsize = 20)
ax.set_yscale("log")
plt.legend(fontsize = 20)
plt.show()


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
sns.lineplot(data=df, x="Year", y="SB_B",hue='Fieldname')
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('year',fontsize = 20)
plt.ylabel('B',fontsize = 20)
ax.set_yscale("log")
plt.show()


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
sns.lineplot(data=df, x="Year", y="SB_T",hue='Fieldname')
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('year',fontsize = 20)
plt.ylabel(r'$𝑡_𝛼$',fontsize = 20)
ax.set_yscale("log")
plt.show()
